# This notebook runs quality control on the data pulled in 001 and ingests it into hd5 files per person

## Set up your environment
Set your Cloud compute profile to: 16 CPUs, 104GB RAM, 1000GB Disk (note that this is at the bottom), and set the Compute type to a Dataproc Cluster. There's a handy link nearby to monitor usage; we like the Yarn dashboard.

Pick 4 Workers and 8 Preemptible workers, each with 16 CPUs, 104 GB RAM and 1000GB Disk.

Go ahead and restart your environment!

### Expected runtime
| Run | Chunks | Approx. time |
|---|---|---|
| TEST = True | chunk 1 (50 participants) | ~2 (estimate) |
| One chunk | ~1,000 participants | ~15–16 min |
| Full run | 60 chunks (~59,000 participants) | ~15–16 hours |

The loop below prints the time for each chunk, so you can estimate a full run from the first chunk. Finished chunks are skipped, so if the run stops you can just rerun the notebook.

## Setup spark

There's a lot of fitbit data to ingest, so we use Dataproc and Spark to parallelize and process faster

In [ ]:
## Install pyspark, h5py and gcsfs (h5py is not preinstalled on the Workbench Dataproc image).
%pip install pyspark h5py gcsfs

In [ ]:
## Import spark libraries
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession
from pyspark.sql.functions import col, collect_list, to_json, struct, to_date, split, date_format
from pyspark.sql import functions as F

In [ ]:
## Import other libraries
import os

# CDR version used by this workspace
os.environ["WORKSPACE_CDR"] = 'wb-silky-artichoke-2408.C2024Q3R9'
import pandas as pd
import time
import h5py
import json
import numpy as np
import gcsfs
from datetime import datetime, timedelta
import subprocess
import shutil
from typing import Dict

In [ ]:
## Set up your spark session -- feel free to name it as you would like. 
## We chose 'qc_ingestion' which is how it will show up in the Yarn dashboard
spark = SparkSession \
  .builder \
  .master('yarn') \
  .appName('qc_ingestion') \
  .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")  # hide Spark's routine warnings

## Paths and settings

In [ ]:
## Workspace bucket paths
# Workspace's own bucket (works in any copy of the workspace)
BUCKET  = subprocess.check_output(['wb', 'resource', 'resolve', '--id=multimodal_risk_prediction']).decode().strip() #gs bucket path
bucket_ = os.path.expanduser('~/workspace/multimodal_risk_prediction') #local fuse mounted path
os.environ["WORKSPACE_BUCKET"] = BUCKET


#paths that are hardcoded to save files generated
file_paths_values = pd.read_csv('filepaths_values.csv')
file_paths_values = file_paths_values[file_paths_values['notebook']=='qc_ingestion'][['notebook','key_','value_']]
paths_dict = file_paths_values.set_index("key_")["value_"]

# TEST = True: quick end-to-end test. 001 pulls only the first 50 EHR participants (chunk 1), and every folder/file name
# from filepaths_values.csv gets a '_test' suffix in 001-005, so a test never overwrites a full run.
# TEST is set once in 001 and read here, so every notebook uses the same value (run 001's setup cell first)
TEST_FLAG_FILE = f'{bucket_}/test_flag.txt'
assert os.path.exists(TEST_FLAG_FILE), "TEST flag not found: run the setup cell in 001 first, it saves TEST for 002-005"
TEST = open(TEST_FLAG_FILE).read().strip() == 'True'
if TEST:
    paths_dict = paths_dict.map(lambda v: v if '.' in str(v) else f'{v}_test')
print(f"TEST={TEST}", dict(paths_dict))

In [ ]:
# Chunks to process. Same split as 001: EHR chunks are 1-37, no-EHR chunks are 38-60.
# N_CHUNKS = None processes every chunk in each group; e.g. N_CHUNKS = 1 does only the first chunk of each group.
N_CHUNKS = None

folder_sleep = paths_dict.get("sleep_import")  # processed sleep data from 001 (same folder for both groups)
GROUPS = [  # (input folder from 001, output folder for hd5 files, chunk numbers)
    (paths_dict.get("ehr_save"),    paths_dict.get("folder_ingest_ehr"),    range(1, 38)),
    (paths_dict.get("no_ehr_save"), paths_dict.get("folder_ingest_no_ehr"), range(38, 61)),
]
if TEST:  # in TEST mode 001 only pulls chunk 1 (50 EHR participants)
    GROUPS = [(GROUPS[0][0], GROUPS[0][1], range(1, 2))]
GROUPS = [(f_in, f_out, list(chunks)[:N_CHUNKS]) for f_in, f_out, chunks in GROUPS]
for f_in, f_out, chunks in GROUPS:
    print(f"{f_in} -> {f_out}: chunks {chunks[0]}-{chunks[-1]} ({len(chunks)})")

## Load data as spark dataframes
For each chunk we read the HR, steps and sleep files from 001 and join them into one row per person.

In [ ]:
def read_tsv(path):
    """Read a tab-separated file from 001 as a spark dataframe."""
    path = path.replace(bucket_, BUCKET)  # Spark workers read from gs://; the mounted folder only exists on the master
    return (spark.read.format("csv")
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .load(path))


def chunk_input_files(no, folder_type):
    """HR, steps and sleep files written by 001 for this chunk."""
    return (f'{bucket_}/{folder_type}/chunk_{no}/fitbit_heart_rate_level_second_chunk_{no}.tsv',
            f'{bucket_}/{folder_type}/chunk_{no}/fitbit_intraday_updated_steps_chunk_{no}.tsv',
            f'{bucket_}/{folder_sleep}/overlapped_entries_df_chunk_{no}.tsv')


def build_person_table(no, folder_type):
    """Join one chunk's HR, steps and sleep data into one row per person (JSON columns)."""
    hr_file, steps_file, sleep_file = chunk_input_files(no, folder_type)
    df = read_tsv(hr_file)        # HR data
    df2 = read_tsv(steps_file)    # steps data
    df3 = read_tsv(sleep_file)    # processed and overlap fixed sleep data
    df3 = df3.withColumn("date", split(col("start_datetime"), " ").getItem(0))

    # aggregate all sleep stages at a day level
    df_inner_agg = (df3.groupBy("person_id", "date")
                    .agg(F.collect_list(
                        F.struct("start_datetime", "end_datetime", "is_main_sleep", "duration_in_min", "level")
                    ).alias("sleep_records_for_date")))
    # aggregate day level sleep to a per person level
    df_outer_agg = (df_inner_agg.groupBy("person_id")
                    .agg(F.collect_list(F.struct("date", "sleep_records_for_date")).alias("sleep_by_date"))
                    .withColumn("sleep_json", F.to_json("sleep_by_date")))

    # join HR and steps at a day level, then aggregate to a per person level
    df = df.join(df2, on=['date', 'person_id'], how='outer')
    result_df = (df.groupBy("person_id")
                 .agg(collect_list(
                     struct("date", "src_id", "sec_timestamps_str", "sec_heart_rate_array_str", "raw_data_count",
                            "heart_rate_array_str", "non_zero_count", "steps_array_str", "non_zero_step_count")
                 ).alias("heartrate_and_steps"))
                 .withColumn("heartrate_and_steps_json", to_json("heartrate_and_steps")))

    # join per person sleep to per person HR and steps
    return df_outer_agg.join(result_df, on=['person_id'], how='outer')

## Quality control functions and constants
### Rules to QC the data
A large part of dealing with any data is understanding how to filter your data. We have decided on several rules which are outlined below. This will ultimately improve our ML models.

**Finalized QC thresholds:**

1. We are replacing unfeasible heart-rate values lower than 30 and higher than 200 with nan
1. We are also replacing unfeasible step count per minute, so the sum of steps do not turn out to be in millions like we had observed in some cases.  We checked the distribution of step counts per minute to find the threshold for the 99th percentile, it was 101, but since during marathon cadence can go to 180-200, we will keep the threshold as 250, and replace anything above that with nan.
1. For flagging a data to be considered or not, we looked at the distribution of wear-time minutes. And found 200 to have a percentile of 1. So we will flag any arrays that have less than 200 minutes of data.  


### Minute array construction rules
Aggregate sleep segment data into per-minute arrays and summary statistics  


Ends at XX:30 → extend to XX+1:00.
Previous ended at XX:30 → next starts at XX+1:00.
Store in HDF5 (compact int mapping)  

We are also saving interpolated heartrate and steps array with sleep time removed, currently we have details for sleep as being in {wake, rem, deep, light, asleep, awake, restless} modes. We will only keep wake or awake time samples, but remove wake, awake time in between sleep.

### QC functions

In [ ]:
# QC thresholds
steps_threshold = 250
hr_low = 30
hr_high = 200
missingness_threshold = 200


# String stage -> compact integer code (for storage)
SLEEP_STAGE_MAPPING: Dict[str, float] = {
    "none": np.nan,
    "deep": 0,
    "wake": 1,
    "light": 2,
    "rem": 3,
    "asleep": 4,
    "restless": 5,
    "awake": 6,
    "unknown": 7,
    # non-main variants
    "nonmain_deep": 8,
    "nonmain_wake": 9,  
    "nonmain_light": 10,
    "nonmain_rem": 11,
    "nonmain_asleep": 12,
    "nonmain_restless": 13,
    "nonmain_awake": 14,
    "nonmain_unknown": 15,
}

# Reverse map for decoding arrays
INT_TO_SLEEP_STAGE: Dict[float, str] = {v: k for k, v in SLEEP_STAGE_MAPPING.items()}


    


# -----------------------------------------------------------------------------
# helper functions
# -----------------------------------------------------------------------------

# Minute array construction with half-minute rules
def create_minute_array_with_precision(segments_df: pd.DataFrame) -> np.ndarray:
    """
    Create a minute-by-minute array with special handling for half-minute transitions:
    - First segment fills normally
    - If a segment ends at XX:30, it fills until XX+1:00
    - Next segment starts at XX+1:00

    Args:
        segments_df: DataFrame with start_datetime, end_datetime, level columns
    Returns:
        NumPy array with minute-by-minute sleep stages
    """
    minute_array = np.zeros(1440, dtype='U20')
    minute_array.fill('none')
    segments_df = segments_df[segments_df['duration_in_min']!=0]
    segments = segments_df.copy().sort_values('start_datetime')

    prev_end_minute = None

    for i, row in segments.iterrows():
        start = row['start_datetime']

        if 'end_datetime' in row and pd.notna(row['end_datetime']):
            end = row['end_datetime']
        else:
            end = start + pd.Timedelta(minutes=row['duration_in_min'])

        if not row['is_main_sleep']:
            add = 'nonmain_'
        else:
            add = ''
        level = add+row['level']

        start_minute_exact = start.hour * 60 + start.minute + start.second/60
        end_minute_exact = end.hour * 60 + end.minute + end.second/60

        # Extract whole minutes and seconds for rule processing
        start_minute = start.hour * 60 + start.minute
        start_second = start.second
        end_minute = end.hour * 60 + end.minute
        end_second = end.second

        # Apply special rules for segments after the first one
        if i > 0 and prev_end_minute is not None and (prev_end_minute.hour!=23 and prev_end_minute.minute!=59):
            # If previous segment ended at XX:30, start this one at XX+1:00
            if prev_end_minute.second == 30:
                rounded_prev_end = prev_end_minute.replace(second=0) + pd.Timedelta(minutes=1)
                # Adjust start time if it's earlier than the rounded end of previous segment
                if start < rounded_prev_end:
                    start = rounded_prev_end
                    start_minute = start.hour * 60 + start.minute
                    start_second = 0

        # Fill the minute array
        start_idx = start_minute

        # If segment ends at XX:30, extend to fill until XX+1:00
        # This applies to ALL segments including the last one
        if end.hour == 0 and end.minute == 0 and end.second == 0:
        # If segment ends at exactly midnight, it should fill until 23:59
            end_idx = 1440 
        else:
            if end_second == 30:
                end_idx = end_minute + 1  # Fill until next minute
            else:
                # Round to nearest minute
                end_idx = end_minute
                if end_second > 0:
                    end_idx += 1

# Ensure bounds
        start_idx = max(0, min(start_idx, 1439))
        end_idx = max(0, min(end_idx, 1440))

        # Fill the array
        minute_array[start_idx:end_idx] = level

        # Store the end time for next iteration
        prev_end_minute = end

    return minute_array


def array_with_fixed_mapping(
    string_array: np.ndarray
) -> np.ndarray:
    """Map a string sleep array (length 1440, one day) to the integer codes in SLEEP_STAGE_MAPPING.
    Unknown labels get 88."""
    int_array = np.zeros(len(string_array), dtype=np.float64)

    # Map each label to its integer code; default to 88 if the label is not in the mapping
    for i, val in enumerate(string_array):
        int_array[i] = SLEEP_STAGE_MAPPING.get(val, 88)

    return int_array
    

### Ingestion with QC
Each person gets two hd5 files: raw data (chunk<no>raw) and QC'd data (chunk<no>_qc), with one group per day. People are processed in parallel on the manager node (one per CPU core, minus 2). Files are written to local disk (~/ingest_tmp) and copied to the bucket after each chunk.

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED

N_WORKERS = max(1, os.cpu_count() - 2)  # people processed in parallel on this machine


def ingest_person(person_id, heartrate_json, sleep_json, raw_directory, qc_directory):
    """QC one person and write their raw + QC hd5 files."""
    # Parse JSON data to Python objects, or use empty lists if None
    heartrate_data = json.loads(heartrate_json) if heartrate_json is not None else []
    sleep_data = json.loads(sleep_json) if sleep_json is not None else []
     # Create a dictionary mapping date strings to sleep records for quick lookup
    sleep_data_by_date = {record['date'][0:10] + ' 00:00:00': record for record in sleep_data}

    # Open two HDF5 files for each person: raw data and quality controlled data (sleep goes in the QC file)
    with h5py.File(f'{raw_directory}/{person_id}.hd5', 'w') as hdf_raw, \
    h5py.File(f'{qc_directory}/{person_id}.hd5', 'w') as hdf_qc: 
        # Process each heart rate and steps record
        for record in heartrate_data:
            date_str = record['date'][0:10] + ' 00:00:00'  # Format date string

            # Create or get groups for this date in each HDF5 file
            raw_date_group = hdf_raw.require_group(date_str)
            qc_date_group = hdf_qc.require_group(date_str)

            src_id = str(record['src_id']) if 'src_id' in record and str(record['src_id']) is not None else 'PTSC' 


            # Process heart rate data if it exists
            if 'heart_rate_array_str' in record and record['heart_rate_array_str'] is not None:
                heart_rate = np.array([float(x) for x in record['heart_rate_array_str'].split(',')])
                if 'sec_heart_rate_array_str' in record and record['sec_heart_rate_array_str'] is not None:
                    heart_rate_sec = np.array([float(x) for x in record['sec_heart_rate_array_str'].split(',')])
                    hr_timestamps_sec = np.array([str(x) for x in record['sec_timestamps_str'].split(',')], dtype=h5py.string_dtype()) 
                    hr_sec_info = float(record['raw_data_count']) 
                else:
                    heart_rate_sec = None

                heart_rate[heart_rate == 0] = np.nan##QC: where fitbit data is missing, 0--np.nan
                heart_rate_p = heart_rate.copy()
                bad_idx_hr = np.where((heart_rate_p > hr_high) | (heart_rate_p < hr_low))[0]
                heart_rate_p[(heart_rate_p > hr_high) | (heart_rate_p < hr_low)] = np.nan ##QC: erroneous values




            else:
                heart_rate = np.array([])
                heart_rate_p = np.array([])
                heart_rate_sec = None
                bad_idx_hr = np.array([])


            if 'steps_array_str' in record and record['steps_array_str'] is not None:
                steps = np.array([float(x) for x in record['steps_array_str'].split(',')])
                steps[(steps == -1)] = np.nan ##QC: where fitbit data is missing, -1--np.nan
                steps_p = steps.copy()
                bad_idx_st = np.where((steps_p > steps_threshold))[0]
                steps_p[(steps_p > steps_threshold)] = np.nan ##QC: erroneous values

            else:
                steps = np.array([])
                steps_p = np.array([])
                bad_idx_st = np.array([])



            # Check if there's too much missing heart rate data after QC
            if len(heart_rate_p) == 0 or \
                np.sum(np.isnan(heart_rate_p))>= missingness_threshold:
                missingness_flag_hr = True
            else:
                missingness_flag_hr = False

            missingness_info_hr = f"{np.sum(np.isnan(heart_rate_p))}/{len(heart_rate_p)}" 


            if len(steps_p) ==0 or \
                np.sum(np.isnan(steps_p))>= missingness_threshold:
                missingness_flag_steps = True
            else:
                missingness_flag_steps = False

            missingness_info_steps = f"{np.sum(np.isnan(steps_p))}/{len(steps_p)}" 


            # Process sleep data - get records from current day
            sleep_record_t = sleep_data_by_date.get(date_str)


            if sleep_record_t:
                sleep_record = (sleep_record_t.get('sleep_records_for_date', []) if sleep_record_t else []) 



                # Convert sleep records to minute-by-minute representation
                if sleep_record:
                    sleep_df_record = pd.DataFrame(sleep_record)
                    sleep_df_record["start_datetime"] = pd.to_datetime(sleep_df_record["start_datetime"],format='ISO8601')
                    sleep_df_record["end_datetime"] = pd.to_datetime(sleep_df_record["end_datetime"],format='ISO8601')

                    sleep_str_array = create_minute_array_with_precision(sleep_df_record)
                    sleep_type_distribution = array_with_fixed_mapping(sleep_str_array)


                else:
                    sleep_type_distribution = np.array([]) 

            else:
                sleep_type_distribution = np.array([]) 






            # Store raw heart rate data in HDF5
            raw_date_group.create_dataset('raw_hr_array', data=heart_rate,compression="gzip")
            raw_date_group.create_dataset('nullified_hr_indices', data=bad_idx_hr,compression="gzip")

            # Store QC'ed heart rate data
            qc_date_group.create_dataset('hr_array_qc', data=heart_rate_p,compression="gzip")

            # Store raw second level heart rate arrays if available
            if heart_rate_sec is not None:
                raw_date_group.create_dataset('raw_hr_array_seconds', data=heart_rate_sec,compression="gzip")
                raw_date_group.create_dataset('raw_hr_available_timestamps_seconds', data=hr_timestamps_sec,compression="gzip")
                raw_date_group.create_dataset('no_of_seconds_of_hr_data', data=hr_sec_info)
                raw_date_group.create_dataset('hr_seconds_array_flag', data=True)

            else:
                raw_date_group.create_dataset('hr_seconds_array_flag', data=False)

            # Store heart rate QC flags and info
            qc_date_group.create_dataset('hr_missingness_flag', data=missingness_flag_hr)
            qc_date_group.create_dataset('hr_missingness_info', data=missingness_info_hr)

# Store sleep data
            qc_date_group.create_dataset('sleep_array', data=sleep_type_distribution,compression="gzip")


            # Store raw steps data in HDF5
            raw_date_group.create_dataset('steps_array', data=steps,compression="gzip")
            raw_date_group.create_dataset('nullified_steps_indices', data=bad_idx_st,compression="gzip")
            # Store QC'ed steps data
            qc_date_group.create_dataset('steps_array_qc', data=steps_p,compression="gzip")
            # Store steps QC flags and info
            qc_date_group.create_dataset('steps_missingness_flag', data=missingness_flag_steps)
            qc_date_group.create_dataset('steps_missingness_info', data=missingness_info_steps)


def ingest_chunk(no, result_df_):
    """QC one chunk and write raw + QC hd5 files per person to local disk. Returns the number of people."""
    raw_directory = os.path.expanduser(f'~/ingest_tmp/chunk_{no}_raw')  # local disk; copied to the bucket in save_chunk
    qc_directory = os.path.expanduser(f'~/ingest_tmp/chunk_{no}_qc')
    os.makedirs(raw_directory, exist_ok=True)
    os.makedirs(qc_directory, exist_ok=True)

    # Spark sends people to this machine one at a time; N_WORKERS of them are processed in parallel
    n_people = 0
    pending = set()
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=multiprocessing.get_context('fork')) as pool:
        for row in result_df_.select("person_id", "heartrate_and_steps_json", "sleep_json").toLocalIterator(prefetchPartitions=True):
            pending.add(pool.submit(ingest_person, row['person_id'], row['heartrate_and_steps_json'],
                                    row['sleep_json'], raw_directory, qc_directory))
            if len(pending) >= 4 * N_WORKERS:  # keep only a few people waiting in memory
                done, pending = wait(pending, return_when=FIRST_COMPLETED)
                for f in done:
                    f.result()  # raises here if a person failed
                    n_people += 1
                    if n_people % 100 == 0:
                        print(f"  chunk {no}: {n_people} people, {time.strftime('%H:%M:%S')}", flush=True)
        for f in pending:
            f.result()
            n_people += 1
    return n_people

### Save data to bucket

In [ ]:
def save_chunk(no, folder_to_save):
    """Copy the chunk's hd5 folders to the bucket, remove local copies, and mark the chunk as done."""
    # gcloud needs the gs:// path; it cannot copy into the mounted folder
    for d in [os.path.expanduser(f'~/ingest_tmp/chunk_{no}_raw'), os.path.expanduser(f'~/ingest_tmp/chunk_{no}_qc')]:
        subprocess.run(f"gcloud storage cp -r --no-user-output-enabled {d}/ {BUCKET}/{folder_to_save}/", shell=True, check=True)
        shutil.rmtree(d)
    # done marker, written only after both copies succeed, so a rerun skips this chunk
    os.makedirs(f'{bucket_}/{folder_to_save}', exist_ok=True)
    open(f'{bucket_}/{folder_to_save}/chunk_{no}.done', 'w').close()

## Run all chunks
Chunks that are already done are skipped, so you can rerun this cell if it stops. Chunks whose 001 files are missing are skipped and listed at the end.

In [ ]:
missing = []
total_start = time.time()
for folder_type, folder_to_save, chunks in GROUPS:
    for no in chunks:
        if os.path.exists(f'{bucket_}/{folder_to_save}/chunk_{no}.done'):
            print(f"chunk {no}: already done, skipping")
            continue
        if not all(os.path.exists(p) for p in chunk_input_files(no, folder_type)):
            print(f"chunk {no}: input files from 001 not found, skipping")
            missing.append(no)
            continue

        t0 = time.time()
        result_df_ = build_person_table(no, folder_type)
        n_people = ingest_chunk(no, result_df_)
        save_chunk(no, folder_to_save)
        print(f"chunk {no}: {n_people} people done in {(time.time() - t0) / 60:.1f} min")

print(f"Finished in {(time.time() - total_start) / 60:.1f} min")
if missing:
    print(f"WARNING: chunks {missing} were skipped because their 001 output is missing. Run 001 for these chunks, then rerun this notebook.")

# Follow along in the '003_v8_data_prepare_for_ML_release' and '004_v8_trends_calculation_release' notebooks for next steps
003 builds the daily ML table and 004 does temporal trend calculations for activity levels from the ingested data